# مرحله ۱-ب: Matcha-TTS در یک محیط پایتون ۳.۱۱ داخل Colab

چرا این نوت‌بوک؟ Colab جدید پایتون ۳.۱۳ می‌دهد، ولی Matcha-TTS مال سال ۲۰۲۴ است و در فهرست وابستگی‌هایش numpy نسخه ۱ را می‌خواهد. روی پایتون ۳.۱۳، pip سراغ numpy نسخه ۱.۲۶.۴ از روی منبع می‌رود و با خطای mesonpy شکست می‌خورد. به‌جای اینکه بی‌نهایت با نسخه‌های قدیمی بجنگیم، یک اتاق کوچک پایتون ۳.۱۱ داخل همان Colab می‌سازیم و Matcha را آنجا نصب می‌کنیم؛ جایی که numpy ۱ بسته آماده دارد.

نکته مهم: این نوت‌بوک، پایتون اصلی Colab را عوض نمی‌کند. فقط یک پوشه به نام matcha311 می‌سازد و با پایتون همان پوشه، سنتز را اجرا می‌کنیم. سلول‌ها را از بالا به پایین و یکی‌یکی اجرا کن.

## قدم ۰: بررسی GPU (دوباره)

اگر Runtime را Restart کرده‌ای، این قدم را دوباره اجرا کن تا مطمئن شوی هنوز T4 داری. خروجی مورد انتظار: CUDA برابر True و نام Tesla T4.

In [ ]:
import sys
import torch

print('Colab Python:', sys.version)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## قدم ۱: نصب espeak-ng و ابزار uv

espeak-ng برای تبدیل متن به واج لازم است. uv هم ابزاری است که پایتون ۳.۱۱ را سریع دانلود و یک محیط مجازی می‌سازد.

In [ ]:
!apt-get update -qq
!apt-get install -y espeak-ng
!which espeak-ng || which espeak || echo 'espeak NOT FOUND'

In [ ]:
!pip install -q uv
!uv --version

## قدم ۲: ساختن محیط پایتون ۳.۱۱

این قدم پایتون ۳.۱۱ را دانلود می‌کند و یک محیط مجازی در پوشه matcha311 می‌سازد. خروجی مورد انتظار در آخرین خط: چیزی شبیه Python 3.11.x.

In [ ]:
!uv python install 3.11
!uv venv --python 3.11 /content/matcha311
!/content/matcha311/bin/python --version

## قدم ۳: نصب torch و Matcha-TTS داخل محیط ۳.۱۱

این قدم طولانی‌ترین قدم است؛ چند دقیقه طول می‌کشد، چون torch با پشتیبانی CUDA و بعد وابستگی‌های Matcha دانلود می‌شوند. صبر کن تا سلول کامل شود و تیک سبز بخورد. در این محیط، numpy نسخه ۱ بسته آماده دارد، پس همان خطای قبلی نباید تکرار شود.

نکته: محیطی که uv می‌سازد به‌صورت پیش‌فرض pip ندارد، پس به‌جای python -m pip از دستور uv pip استفاده می‌کنیم که مستقیم داخل همان محیط نصب می‌کند.


In [ ]:
!uv pip install --python /content/matcha311/bin/python --upgrade pip
!uv pip install --python /content/matcha311/bin/python torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!uv pip install --python /content/matcha311/bin/python matcha-tts

## قدم ۴: بررسی نصب داخل محیط ۳.۱۱

این قدم سه چیز را چک می‌کند: نسخه پایتون محیط، اینکه torch داخل همان محیط GPU را می‌بیند، و اینکه دستور matcha-tts وجود دارد. خروجی مورد انتظار: torch با CUDA برابر True و نام Tesla T4.

In [ ]:
!/content/matcha311/bin/python -c "import torch; print('torch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')"
!/content/matcha311/bin/python -c "import numpy; print('numpy:', numpy.__version__)"
!/content/matcha311/bin/matcha-tts --help | head -n 20

## قدم ۴-ب: وصله بارگذاری چک‌پوینت (فقط برای چک‌پوینت رسمی Matcha)

PyTorch از نسخه ۲.۶ به بعد به‌صورت پیش‌فرض فقط وزن‌های ساده را با weights_only=True بار می‌کند، ولی چک‌پوینت قدیمی Matcha داخل خودش یک شیء پیکربندی omegaconf دارد و با این حالت رد می‌شود. چون این چک‌پوینت به‌صورت خودکار از انتشار رسمی خود Matcha دانلود شده و منبع مطمئنی است، در تابع load_matcha گزینه weights_only=False را اضافه می‌کنیم. این گزینه را هرگز برای فایلی از یک منبع ناشناس استفاده نکن.


In [ ]:
!sed -i "s/MatchaTTS.load_from_checkpoint(checkpoint_path, map_location=device)/MatchaTTS.load_from_checkpoint(checkpoint_path, map_location=device, weights_only=False)/" /content/matcha311/lib/python3.11/site-packages/matcha/cli.py
!grep -n "load_from_checkpoint" /content/matcha311/lib/python3.11/site-packages/matcha/cli.py


## قدم ۵: یک جمله آزمایشی با ۱۰ گام

جمله ثابت مرحله ۱:

> Hello, this is a test of Matcha text to speech with ten steps.

در اولین اجرا، مدل آماده matcha_ljspeech و وکودر hifigan_T2_v1 دانلود می‌شوند. خروجی باید utterance_001.wav باشد.

In [ ]:
!/content/matcha311/bin/matcha-tts --text "Hello, this is a test of Matcha text to speech with ten steps." --steps 10 --output_folder step1_outputs
!ls -lh step1_outputs

## قدم ۶: گوش دادن و بررسی فایل

این سلول در پایتون اصلی Colab اجرا می‌شود و فقط فایل ساخته‌شده را می‌خواند و پخش می‌کند. اگر صدا قابل فهم بود، مرحله ۱ قبول است.

In [ ]:
import contextlib
import os
import wave

from IPython.display import Audio, display

wav_path = 'step1_outputs/utterance_001.wav'
assert os.path.exists(wav_path), f'Not found: {wav_path}. Step 5 did not produce audio.'
with contextlib.closing(wave.open(wav_path)) as w:
    duration = w.getnframes() / w.getframerate()
    sample_rate = w.getframerate()
print(f'sample rate: {sample_rate} Hz')
print(f'duration: {duration:.2f} s')
print(f'size: {os.path.getsize(wav_path) / 1024:.1f} KB')
display(Audio(wav_path))

## چک‌لیست موفقیت

- [ ] قدم ۰: CUDA برابر True و GPU برابر Tesla T4.
- [ ] قدم ۲: نسخه پایتون محیط برابر ۳.۱۱.x.
- [ ] قدم ۴: torch داخل محیط، CUDA برابر True را نشان داد.
- [ ] قدم ۵: فایل step1_outputs/utterance_001.wav ساخته شد.
- [ ] قدم ۶: صدا پخش شد و جمله قابل فهم بود.

## اگر خطا گرفتی

در همان قدمی که خطا داده توقف کن و ۳۰ خط آخر خروجی قرمز همان قدم را بفرست. مخصوصا این موارد را جداگانه بگو: در کدام قدم بود، و آیا قدم‌های قبلش تیک سبز خورده بودند یا نه.

## مرحله بعد (هنوز اجرا نکن)

مرحله ۲: با همین دستور محیط ۳.۱۱، همان جمله‌ها را با steps برابر ۱، ۲، ۴، ۱۰ و ۲۵ تولید می‌کنیم و RTF و WER را اندازه می‌گیریم.